# HLS Video Player — Google Colab 起動ノート

このノートブックは hls-video-player を Colab 上で動作させ、Google Drive を永続化領域として使います。

1. FFmpeg インストール
2. Python 依存インストール
3. Google Drive マウント
4. リポジトリ取得
5. `MEDIA_ROOT` を Drive に設定
6. `share=True` でアプリ起動（*.gradio.live の公開 URL）

## 1. FFmpeg インストール

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!ffmpeg -version | head -1

## 2. Python 依存インストール

In [ ]:
!pip install -q gradio fastapi uvicorn python-multipart

## 3. Google Drive マウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. リポジトリ取得

社内 Gitea (motowo/hls-video-player) から取得。ngrok / Cloudflare Tunnel 等で Gitea が外部から見える必要があります。
外部 Git ホスティング（GitHub ミラー等）にしている場合は URL を書き換えてください。

In [ ]:
import os
REPO_URL = os.environ.get('HLS_REPO_URL', 'https://github.com/YOUR/hls-video-player.git')
!test -d hls-video-player || git clone {REPO_URL} hls-video-player
%cd hls-video-player
!pip install -q -e .

## 5. `MEDIA_ROOT` を Drive に設定

Drive 配下に `hls-video/media/{source,hls,sprites}` を作成（既にある場合は何もしない）。
Drive はローカル VM ストレージより遅いので、大きな動画の変換は時間が伸びる点に注意。

In [ ]:
import os

MEDIA = '/content/drive/MyDrive/hls-video/media'
for sub in ('source', 'hls', 'sprites'):
    os.makedirs(f'{MEDIA}/{sub}', exist_ok=True)

os.environ['MEDIA_ROOT'] = MEDIA
os.environ['MAX_CONCURRENT_JOBS'] = '1'      # Colab は通常 2 vCPU
os.environ['FFMPEG_THREADS'] = '2'
os.environ['FFMPEG_PRESET'] = 'veryfast'
os.environ['GRADIO_SHARE'] = 'true'          # share=True 用
os.environ['PORT'] = '7860'

!ls {MEDIA}

## 6. 起動

Gradio の `share=True` で *.gradio.live のトンネル URL が表示される。
`/hls/*` や `/sprites/*` も同じトンネル経由で配信される（FastAPI マウントのため）。

ランタイム切断時は Drive 上の変換結果は残るので、再接続後にこのノートを再実行すれば復帰できます。

In [ ]:
# 起動方法 A: Gradio の組み込みサーバで起動（share=True 対応）
import gradio as gr
from fastapi import FastAPI
from app.gradio_ui import build_ui
from app.static_mount import mount_static
from app.player_embed import router as player_router
from hls_video.config import media_root

media = media_root()
fapi = FastAPI()
mount_static(fapi, media_root=str(media), static_dir='/content/hls-video-player/static')
fapi.include_router(player_router)

demo = build_ui(media_dir=media)
demo.queue()
gr.mount_gradio_app(fapi, demo, path='/')

# Colab では別途 uvicorn を起動し、share の public URL は demo.launch() の戻りから取得
import uvicorn, threading
server_thread = threading.Thread(
    target=lambda: uvicorn.run(fapi, host='0.0.0.0', port=7860, log_level='warning'),
    daemon=True,
)
server_thread.start()
print('Local: http://localhost:7860')

# share URL はセル出力からクリックしてください
# （demo.launch(share=True) を別セルで呼んで tunnel を張る場合は下のセルを使用）

### 代替: `demo.launch(share=True)` による公開 URL

上のセルはローカル起動のみ。Colab の外部からアクセスしたい場合は次のセルを代わりに実行してください。
この場合 FastAPI の追加マウント（/hls, /sprites 等）も同じトンネルで配信されます。

In [ ]:
# from app.main import build_app
# import uvicorn
# uvicorn.run(build_app(), host='0.0.0.0', port=7860)
#
# または gradio の share:
# from app.gradio_ui import build_ui
# demo = build_ui()
# demo.launch(share=True)